# Electricity Consumption Prediction
Estimate daily household electricity use in kWh using a supervised regression workflow.

**Dataset note:** This is synthetic practice data, not real smart-meter records.

## Part A - Problem Understanding
Predict a numerical target (daily electricity consumption in kWh), so this is a supervised regression problem.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
p=Path("../data/electricity_consumption.csv")
if not p.exists(): p=Path("data/electricity_consumption.csv")
df=pd.read_csv(p)
df.head()

## Part B - Dataset Understanding

In [ ]:
print("Shape:",df.shape)
print(df.dtypes)
print("Missing values:\n",df.isnull().sum())
print("Duplicate rows:",df.duplicated().sum())
df.describe()

## Part C - Data Preprocessing
Remove duplicates and check missing values. Selected inputs are numeric; Weekend is already encoded as 0 or 1.

In [ ]:
df=df.drop_duplicates().dropna().copy()
print("Cleaned shape:",df.shape)

## Part D - Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(7,4)); sns.scatterplot(data=df,x="AC_Hours",y="Daily_Consumption_kWh"); plt.title("AC Hours vs Consumption"); plt.show()
plt.figure(figsize=(7,4)); sns.scatterplot(data=df,x="Temperature_C",y="Daily_Consumption_kWh"); plt.title("Temperature vs Consumption"); plt.show()
plt.figure(figsize=(8,5)); sns.heatmap(df.corr(numeric_only=True),annot=True,cmap="YlGnBu",fmt=".2f"); plt.title("Correlation Heatmap"); plt.show()
plt.figure(figsize=(7,4)); sns.histplot(df["Daily_Consumption_kWh"],kde=True); plt.title("Daily Consumption Distribution"); plt.show()

### EDA observations
1. AC usage is designed to have a positive relationship with consumption.
2. Higher temperatures can increase cooling demand.
3. The heatmap compares numerical relationships.
4. The histogram shows the distribution of consumption values.
These are patterns in synthetic data, not measured household findings.

## Part E - Features and Target

In [ ]:
features=["Temperature_C","Humidity_Percent","Occupants","AC_Hours","Appliance_Hours","Lighting_Hours","Weekend"]
X=df[features]; y=df["Daily_Consumption_kWh"]
print(X.shape,y.shape)

## Part F - Train-Test Split (80/20)

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
print(len(X_train),len(X_test))

## Parts G and H - Model Selection and Training

In [ ]:
models={"Linear Regression":LinearRegression(),"Decision Tree":DecisionTreeRegressor(max_depth=6,random_state=42),"Random Forest":RandomForestRegressor(n_estimators=160,random_state=42)}
trained={}
for name,m in models.items():
    m.fit(X_train,y_train); trained[name]=m
print("Models trained.")

## Part I - Evaluation

In [ ]:
rows=[]
for name,m in trained.items():
    pred=m.predict(X_test); mse=mean_squared_error(y_test,pred)
    rows.append({"Model":name,"MAE":mean_absolute_error(y_test,pred),"MSE":mse,"RMSE":np.sqrt(mse),"R2":r2_score(y_test,pred)})
results=pd.DataFrame(rows).sort_values("RMSE")
results.round(3)

## Part J - Prediction Visualizations

In [ ]:
best_name=results.iloc[0]["Model"]; best=trained[best_name]; pred=best.predict(X_test)
plt.figure(figsize=(6,5)); plt.scatter(y_test,pred,alpha=.7); plt.xlabel("Actual kWh"); plt.ylabel("Predicted kWh"); plt.title("Actual vs Predicted"); plt.show()
plt.figure(figsize=(7,4)); sns.histplot(y_test-pred,kde=True); plt.title("Residual Distribution"); plt.show()
plt.figure(figsize=(7,4)); sns.barplot(data=results,x="Model",y="RMSE"); plt.title("RMSE Model Comparison"); plt.xticks(rotation=15); plt.show()

## Part K - Improvement Experiments

In [ ]:
experiments=[]
for depth in [3,5,8,None]:
    m=DecisionTreeRegressor(max_depth=depth,random_state=42); m.fit(X_train,y_train); p=m.predict(X_test)
    experiments.append({"Setting":f"Tree depth {depth}","RMSE":np.sqrt(mean_squared_error(y_test,p))})
for count in [50,100,200]:
    m=RandomForestRegressor(n_estimators=count,random_state=42); m.fit(X_train,y_train); p=m.predict(X_test)
    experiments.append({"Setting":f"Forest trees {count}","RMSE":np.sqrt(mean_squared_error(y_test,p))})
pd.DataFrame(experiments).sort_values("RMSE").round(3)

## Part L - Interpretation and Saving

In [ ]:
print("Selected model:",best_name)
print(results.iloc[0].round(3))
out=Path("../model")
if not out.exists(): out=Path("model")
out.mkdir(exist_ok=True); joblib.dump(best,out/"electricity_model.pkl")
sample=X_test.iloc[[0]]
print("Sample estimate (kWh):",round(float(best.predict(sample)[0]),2))